# Analysis of Shuffled Simulation Progress

This notebook analyzes the progress of shuffled simulations for GABA and Myelin receptor maps.

## Structure
- **GABA simulations**: `/home/frank/HBNM/outputs/gaba/`
- **Myelin simulations**: `/home/frank/HBNM/outputs/myelin/`

Each simulation type contains multiple surrogate subfolders, with each surrogate containing:
- Multiple iteration HDF5 files (`iteration_1.hdf5`, `iteration_2.hdf5`, etc.)
- Sample NPZ files (`samples_1.npz`, etc.)

## Analysis Includes
1. **Data Loading**: Custom loading function for shuffled simulation structure
2. **Summary Statistics**: Final acceptance rates, epsilon values, and best distances
3. **Progress Analysis**: Identification of problematic vs. well-performing surrogates
4. **Comparative Analysis**: GABA vs. Myelin performance comparison

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from hbnm.io import Data
from hbnm.bnm import Bnm
from hbnm.model.utils import subdiag, fisher_z
from hbnm.analysis import SimulationAnalyzer
from scipy.stats import pearsonr
import os
import pandas as pd

# Set up matplotlib for better plots
plt.style.use('default')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300


Assess Simulations

In [2]:
gaba_path = '/home/frank/HBNM/outputs/gaba'
myelin_path = '/home/frank/HBNM/outputs/myelin'

In [3]:
# Get the current notebook directory
notebook_dir = os.getcwd()

# Navigate to the project root (HBNM directory)
project_root = os.path.dirname(notebook_dir)  # Goes up one level from experiments to HBNM

# Set up data directory using os.path.join for cross-platform compatibility
data_dir = os.path.join(project_root, 'data/')
maps_dir = os.path.join(data_dir, 'maps')
output_dir = os.path.join(project_root, 'outputs')

data = Data(data_dir, output_dir)
analyzer = SimulationAnalyzer(data, verbose=True)

sims = [
    'gaba',
    'myelin', 
]

In [4]:
import h5py
import glob
from collections import defaultdict

def load_shuffled_simulation_data(base_path, sim_name):
    """Load data from shuffled simulation structure"""
    simulation_data = {
        'surrogates': {},
        'summary_stats': {}
    }
    
    # Get all surrogate folders
    surrogate_folders = sorted(glob.glob(os.path.join(base_path, 'surrogate_*')))
    print(f"Found {len(surrogate_folders)} surrogate folders for {sim_name}")
    
    all_final_acceptance_rates = []
    all_final_epsilons = []
    all_best_distances = []
    all_iterations_counts = []
    
    for surrogate_folder in surrogate_folders:
        surrogate_name = os.path.basename(surrogate_folder)
        
        # Get all iteration files for this surrogate
        iteration_files = sorted(glob.glob(os.path.join(surrogate_folder, 'iteration_*.hdf5')),
                                key=lambda x: int(x.split('_')[-1].split('.')[0]))
        
        if not iteration_files:
            continue
            
        surrogate_data = {
            'n_iterations': len(iteration_files),
            'iteration_files': iteration_files
        }
        
        # Load data from the final iteration
        final_iteration_file = iteration_files[-1]
        
        try:
            with h5py.File(final_iteration_file, 'r') as f:
                # Calculate acceptance rate
                n_accepted = f['n_accepted'][()]
                n_total = f['n_total'][()]
                final_acceptance_rate = n_accepted / n_total if n_total > 0 else 0
                
                # Get final epsilon
                final_epsilon = f['epsilon'][()]
                
                # Calculate best distance across all iterations for this surrogate
                best_distances_per_iteration = []
                for iteration_file in iteration_files:
                    with h5py.File(iteration_file, 'r') as f_iter:
                        distances = f_iter['distance'][:]
                        best_distances_per_iteration.append(np.min(distances))
                
                best_distance = np.min(best_distances_per_iteration)
                
                # Store surrogate data
                surrogate_data.update({
                    'final_acceptance_rate': final_acceptance_rate,
                    'final_epsilon': final_epsilon,
                    'best_distance': best_distance,
                    'best_distances_per_iteration': best_distances_per_iteration
                })
                
                # Collect for overall statistics
                all_final_acceptance_rates.append(final_acceptance_rate)
                all_final_epsilons.append(final_epsilon)
                all_best_distances.append(best_distance)
                all_iterations_counts.append(len(iteration_files))
                
        except Exception as e:
            print(f"Error loading {surrogate_name}: {e}")
            continue
            
        simulation_data['surrogates'][surrogate_name] = surrogate_data
    
    # Calculate summary statistics
    if all_final_acceptance_rates:
        simulation_data['summary_stats'] = {
            'n_surrogates': len(all_final_acceptance_rates),
            'iterations_range': (min(all_iterations_counts), max(all_iterations_counts)),
            'avg_iterations': np.mean(all_iterations_counts),
            'final_acceptance_rate': {
                'mean': np.mean(all_final_acceptance_rates),
                'std': np.std(all_final_acceptance_rates),
                'min': np.min(all_final_acceptance_rates),
                'max': np.max(all_final_acceptance_rates)
            },
            'final_epsilon': {
                'mean': np.mean(all_final_epsilons),
                'std': np.std(all_final_epsilons),
                'min': np.min(all_final_epsilons),
                'max': np.max(all_final_epsilons)
            },
            'best_distance': {
                'mean': np.mean(all_best_distances),
                'std': np.std(all_best_distances),
                'min': np.min(all_best_distances),
                'max': np.max(all_best_distances)
            }
        }
    
    return simulation_data

# Load shuffled simulation data
print("Loading shuffled simulation data...")
print("=" * 80)

# Load GABA data
print("\nLoading GABA simulations...")
gaba_data = load_shuffled_simulation_data(gaba_path, 'GABA')

print("\nLoading Myelin simulations...")
myelin_data = load_shuffled_simulation_data(myelin_path, 'Myelin')

# Display comprehensive summary
print("\n" + "=" * 80)
print("SHUFFLED SIMULATION SUMMARY")
print("=" * 80)

for sim_name, sim_data in [('GABA', gaba_data), ('MYELIN', myelin_data)]:
    print(f"\n{sim_name}:")
    stats = sim_data['summary_stats']
    
    if stats:
        print(f"  • Surrogates analyzed: {stats['n_surrogates']}")
        print(f"  • Iterations per surrogate: {stats['iterations_range'][0]}-{stats['iterations_range'][1]} (avg: {stats['avg_iterations']:.1f})")
        print(f"  • Final acceptance rate: {stats['final_acceptance_rate']['mean']:.1%} ± {stats['final_acceptance_rate']['std']:.1%}")
        print(f"    Range: {stats['final_acceptance_rate']['min']:.1%} - {stats['final_acceptance_rate']['max']:.1%}")
        print(f"  • Final epsilon: {stats['final_epsilon']['mean']:.4f} ± {stats['final_epsilon']['std']:.4f}")
        print(f"    Range: {stats['final_epsilon']['min']:.4f} - {stats['final_epsilon']['max']:.4f}")
        print(f"  • Best distance achieved: {stats['best_distance']['mean']:.4f} ± {stats['best_distance']['std']:.4f}")
        print(f"    Range: {stats['best_distance']['min']:.4f} - {stats['best_distance']['max']:.4f}")
        print(f"    Overall best: {stats['best_distance']['min']:.4f}")
    else:
        print("  No data loaded")

Loading shuffled simulation data...

Loading GABA simulations...
Found 68 surrogate folders for GABA

Loading Myelin simulations...
Found 100 surrogate folders for Myelin

SHUFFLED SIMULATION SUMMARY

GABA:
  • Surrogates analyzed: 68
  • Iterations per surrogate: 7-50 (avg: 41.9)
  • Final acceptance rate: 24.4% ± 19.0%
    Range: 1.3% - 80.6%
  • Final epsilon: 0.5472 ± 0.0538
    Range: 0.4224 - 0.6391
  • Best distance achieved: 0.5347 ± 0.0448
    Range: 0.4135 - 0.6211
    Overall best: 0.4135

MYELIN:
  • Surrogates analyzed: 100
  • Iterations per surrogate: 1-51 (avg: 12.5)
  • Final acceptance rate: 72.0% ± 34.0%
    Range: 1.6% - 100.0%
  • Final epsilon: 0.6362 ± 0.0618
    Range: 0.4615 - 0.6904
  • Best distance achieved: 0.5773 ± 0.0491
    Range: 0.4612 - 0.6431
    Overall best: 0.4612


In [5]:
# Create summary report
print("\n" + "="*60)
print("COMPREHENSIVE SUMMARY REPORT")
print("="*60)

summary_df = analyzer.create_summary_report()
print(summary_df.to_string(index=False))

# Save summary to CSV
# summary_df.to_csv(f"{output_dir}/receptor_simulation_summary.csv", index=False)
# print(f"\nSummary saved to: {output_dir}/receptor_simulation_summary.csv")


DETAILED SURROGATE ANALYSIS

Showing first 5 and last 5 surrogates from each simulation:
Total surrogates: GABA=68, MYELIN=100

GABA Sample Results:
    Surrogate  Iterations Final_Acceptance_Rate Final_Epsilon Best_Distance
surrogate_001          50                 15.7%        0.5020        0.4945
surrogate_002          50                 36.8%        0.5219        0.5216
surrogate_003          36                 78.1%        0.6284        0.6187
surrogate_067          50                 27.8%        0.4593        0.4584
surrogate_068           9                 52.1%        0.5986        0.5529

MYELIN Sample Results:
    Surrogate  Iterations Final_Acceptance_Rate Final_Epsilon Best_Distance
surrogate_001          51                  8.1%        0.5065        0.5029
surrogate_002          50                  1.8%        0.5262        0.5144
surrogate_003          50                 17.1%        0.5497        0.5492
surrogate_099           1                100.0%        0.6697     

In [6]:
# Progress tracking and identification of problematic surrogates
print("\n" + "=" * 80)
print("PROGRESS ANALYSIS")
print("=" * 80)

def analyze_progress_issues(sim_data, sim_name, acceptance_threshold=0.05, epsilon_threshold=0.8):
    """Identify surrogates with potential issues"""
    issues = {
        'low_acceptance': [],
        'high_epsilon': [],
        'few_iterations': [],
        'good_surrogates': []
    }
    
    for surrogate_name, surrogate_info in sim_data['surrogates'].items():
        if surrogate_info['final_acceptance_rate'] < acceptance_threshold:
            issues['low_acceptance'].append((surrogate_name, surrogate_info['final_acceptance_rate']))
        
        if surrogate_info['final_epsilon'] > epsilon_threshold:
            issues['high_epsilon'].append((surrogate_name, surrogate_info['final_epsilon']))
            
        if surrogate_info['n_iterations'] < 30:
            issues['few_iterations'].append((surrogate_name, surrogate_info['n_iterations']))
            
        if (surrogate_info['final_acceptance_rate'] >= acceptance_threshold and 
            surrogate_info['final_epsilon'] <= epsilon_threshold and 
            surrogate_info['n_iterations'] >= 30):
            issues['good_surrogates'].append((surrogate_name, surrogate_info['best_distance']))
    
    return issues

# Analyze both simulations
for sim_name, sim_data in [('GABA', gaba_data), ('MYELIN', myelin_data)]:
    print(f"\n{sim_name} Progress Analysis:")
    issues = analyze_progress_issues(sim_data, sim_name)
    
    print(f"  • Surrogates with low acceptance (<5%): {len(issues['low_acceptance'])}")
    if issues['low_acceptance'][:3]:  # Show first 3
        for name, rate in issues['low_acceptance'][:3]:
            print(f"    - {name}: {rate:.1%}")
        if len(issues['low_acceptance']) > 3:
            print(f"    ... and {len(issues['low_acceptance']) - 3} more")
    
    print(f"  • Surrogates with high epsilon (>0.8): {len(issues['high_epsilon'])}")
    if issues['high_epsilon'][:3]:  # Show first 3
        for name, eps in issues['high_epsilon'][:3]:
            print(f"    - {name}: {eps:.4f}")
        if len(issues['high_epsilon']) > 3:
            print(f"    ... and {len(issues['high_epsilon']) - 3} more")
    
    print(f"  • Surrogates with few iterations (<30): {len(issues['few_iterations'])}")
    if issues['few_iterations'][:3]:  # Show first 3
        for name, iters in issues['few_iterations'][:3]:
            print(f"    - {name}: {iters} iterations")
        if len(issues['few_iterations']) > 3:
            print(f"    ... and {len(issues['few_iterations']) - 3} more")
    
    print(f"  • Well-performing surrogates: {len(issues['good_surrogates'])}")
    if issues['good_surrogates']:
        # Sort by best distance and show top 3
        issues['good_surrogates'].sort(key=lambda x: x[1])
        for name, dist in issues['good_surrogates'][:3]:
            print(f"    - {name}: distance {dist:.4f}")
        if len(issues['good_surrogates']) > 3:
            print(f"    ... and {len(issues['good_surrogates']) - 3} more")



PROGRESS ANALYSIS

GABA Progress Analysis:
  • Surrogates with low acceptance (<5%): 6
    - surrogate_026: 1.3%
    - surrogate_029: 4.7%
    - surrogate_031: 3.1%
    ... and 3 more
  • Surrogates with high epsilon (>0.8): 0
  • Surrogates with few iterations (<30): 15
    - surrogate_009: 26 iterations
    - surrogate_014: 12 iterations
    - surrogate_022: 13 iterations
    ... and 12 more
  • Well-performing surrogates: 47
    - surrogate_036: distance 0.4135
    - surrogate_048: distance 0.4206
    - surrogate_053: distance 0.4346
    ... and 44 more

MYELIN Progress Analysis:
  • Surrogates with low acceptance (<5%): 6
    - surrogate_002: 1.8%
    - surrogate_004: 1.9%
    - surrogate_013: 3.1%
    ... and 3 more
  • Surrogates with high epsilon (>0.8): 0
  • Surrogates with few iterations (<30): 78
    - surrogate_008: 12 iterations
    - surrogate_017: 15 iterations
    - surrogate_019: 14 iterations
    ... and 75 more
  • Well-performing surrogates: 16
    - surrogate_009:

In [7]:
# Final summary and recommendations
print("\n" + "=" * 80)
print("FINAL SUMMARY & RECOMMENDATIONS")
print("=" * 80)

# Compare GABA vs MYELIN
gaba_stats = gaba_data['summary_stats']
myelin_stats = myelin_data['summary_stats']

print("\nComparative Summary:")
print(f"{'Metric':<25} {'GABA':<20} {'MYELIN':<20}")
print("-" * 65)
print(f"{'Surrogates completed':<25} {gaba_stats['n_surrogates']:<20} {myelin_stats['n_surrogates']:<20}")
print(f"{'Avg iterations':<25} {gaba_stats['avg_iterations']:<20.1f} {myelin_stats['avg_iterations']:<20.1f}")
print(f"{'Avg acceptance rate':<25} {gaba_stats['final_acceptance_rate']['mean']:<20.1%} {myelin_stats['final_acceptance_rate']['mean']:<20.1%}")
print(f"{'Avg final epsilon':<25} {gaba_stats['final_epsilon']['mean']:<20.4f} {myelin_stats['final_epsilon']['mean']:<20.4f}")
print(f"{'Best distance overall':<25} {gaba_stats['best_distance']['min']:<20.4f} {myelin_stats['best_distance']['min']:<20.4f}")

print(f"\nRecommendations:")
print("1. Focus on surrogates with consistently low acceptance rates for debugging")
print("2. Consider extending iterations for surrogates that stopped early (<30 iterations)")
print("3. Monitor epsilon convergence - values >0.8 may indicate fitting difficulties")
print(f"4. GABA best result: {gaba_stats['best_distance']['min']:.4f}")
print(f"5. MYELIN best result: {myelin_stats['best_distance']['min']:.4f}")

# Save summary to file for reference
summary_dict = {
    'GABA': gaba_stats,
    'MYELIN': myelin_stats,
    'analysis_timestamp': pd.Timestamp.now().isoformat()
}

# Optional: Save detailed data
# combined_df.to_csv(f"{output_dir}/shuffled_simulation_detailed_summary.csv", index=False)
# print(f"\nDetailed results saved to: {output_dir}/shuffled_simulation_detailed_summary.csv")

print(f"\nAnalysis complete! Total surrogates analyzed: {gaba_stats['n_surrogates'] + myelin_stats['n_surrogates']}")



FINAL SUMMARY & RECOMMENDATIONS

Comparative Summary:
Metric                    GABA                 MYELIN              
-----------------------------------------------------------------
Surrogates completed      68                   100                 
Avg iterations            41.9                 12.5                
Avg acceptance rate       24.4%                72.0%               
Avg final epsilon         0.5472               0.6362              
Best distance overall     0.4135               0.4612              

Recommendations:
1. Focus on surrogates with consistently low acceptance rates for debugging
2. Consider extending iterations for surrogates that stopped early (<30 iterations)
3. Monitor epsilon convergence - values >0.8 may indicate fitting difficulties
4. GABA best result: 0.4135
5. MYELIN best result: 0.4612

Analysis complete! Total surrogates analyzed: 168
